In [2]:
import pandas as pd

Here I load the movie data again (metaclean_v2.csv), because I need the cast column and the movie_id to build the ACTOR table. The movie.csv doesn't have the cast column anymore, so I can't use that one for this.

In [3]:
df = pd.read_csv('metaclean_v2.csv')


In the cast column all the actors of a movie are in one cell, separated by a comma. That doesn't work for a database, so here I split the cast on the comma and gave every actor of a movie its own row, together with the movie_id. I also stripped the spaces around the names. There are 3,675 movies without a cast, and these drop out here, because there is nothing to link them to.

In [4]:
exploded = (
    df[["movie_id", "cast"]]
    .assign(cast=df["cast"].str.split(","))
    .explode("cast")
)
exploded["cast"] = exploded["cast"].str.strip()
exploded = exploded.dropna(subset=["cast"])

Before building the table I did a check on the split data, and it turns out there are two small problems. There are 18 empty names, which are probably caused by an extra comma in the original data, and in 12 cases the same actor is listed twice for the same movie. The duplicates are a problem for the junction table later, because the combination of movie_id and actor_id has to be unique if I want to use it as a primary key. So I removed both, which brings the data from 48,293 down to 48,263 rows.

In [5]:
print(len(exploded), "rows before cleaning")
print((exploded["cast"] == "").sum(), "empty names")
print(exploded.duplicated().sum(), "duplicate rows")

exploded = exploded[exploded["cast"] != ""]
exploded = exploded.drop_duplicates()

print(len(exploded), "rows after cleaning")

48293 rows before cleaning
18 empty names
12 duplicate rows
48263 rows after cleaning


Now I took all the unique actor names and gave each of them their own actor_id (1, 2, 3 etc.), which gives me the ACTOR table with actor_id and actor_name. This resulted in 24,844 unique actors, and I checked that every actor_id is unique. One thing to keep in mind is that the data only has names, so two different actors with the same name get the same actor_id. For our research question this is not a problem, but it is a small limitation. In the end I saved the table as actor.csv.

In [6]:
unique_actors = sorted(exploded["cast"].unique())
actor = pd.DataFrame({
    "actor_id": range(1, len(unique_actors) + 1),
    "actor_name": unique_actors,
})

actor.to_csv("actor.csv", index=False)

print(actor.shape)
print(actor["actor_id"].is_unique)

(24844, 2)
True


Now I build the actor_connect table, which links every movie to its actors. I took the split cast data from the previous step (one actor per row, with the movie_id) and merged it with the ACTOR table on the actor name, so every row gets the right actor_id. After that I only kept movie_id and actor_id, because the name already lives in the ACTOR table.

In [8]:
actor_connect = exploded.merge(actor, left_on="cast", right_on="actor_name", how="left")
actor_connect = actor_connect[["movie_id", "actor_id"]]

Here I checked the result. The table has 48,263 rows, which is the same as the cleaned split data, so no rows got lost or doubled in the merge. Every row got an actor_id, there are no duplicate rows, so the combination of movie_id and actor_id is unique. The table links 7,657 movies to 24,844 actors. That means that all the actors from the ACTOR table are used, and that the 3,675 movies without a cast are not in this table.

In [9]:
print(actor_connect.shape)
print(actor_connect["actor_id"].isna().sum())
print(actor_connect.duplicated().sum())
print(actor_connect["movie_id"].nunique(), "movies with at least one actor")
print(actor_connect["actor_id"].nunique(), "actors that are linked to a movie")

(48263, 2)
0
0
7657 movies with at least one actor
24844 actors that are linked to a movie


In the end I saved the junction table as actor_connect.csv.

In [10]:
actor_connect.to_csv("actor_connect.csv", index=False)